# DenseNet E2: ASL và so sánh với weighted BCE

Bản sao từ `model/CheXpert_relation_aware_Pneumonia_Pneumothorax.ipynb`.
Giữ kiến trúc E2, seed 42, patient-level split 80/10/10, ảnh 320, batch 16,
Adam, learning rate 1e-4, 5 epoch và uncertainty policies của notebook gốc.
Thay `weighted_bce` bằng **ASL cho binary heads**; Cardiomegaly và Pleural
Effusion vẫn dùng cross-entropy 3 lớp (`u_multiclass`). ASL không dùng `pos_weight`.

ASL: gamma_neg=4, gamma_pos=1, clip=0.05. Công thức theo
[triển khai chính thức](https://github.com/Alibaba-MIIL/ASL/blob/main/src/loss_functions/losses.py),
giữ gradient qua focal weight như nhánh ASL đã có trong notebook gốc
(tương ứng `disable_torch_grad_focal_loss=False`).

## Chạy trên Kaggle

1. Import notebook này, bật GPU, gắn dataset CheXpert `ashery/chexpert` như bản gốc.
2. Gắn thêm model/dataset chứa `best_E2_relation_aware_weighted_bce.pt`.
   Cell cấu hình tìm đúng tên file trong `/kaggle/input`; nếu có nhiều bản,
   đặt `BCE_CHECKPOINT_PATH` trỏ tới bản cần so sánh.
3. Bật Internet để torchvision tải ImageNet weights nếu chưa có trong cache.
4. Restart kernel rồi **Run All**. Kết quả nằm trong thư mục mới
   `/kaggle/working/chexpert_e2_asl_<UTC timestamp>/`.

Trên máy riêng, đặt biến môi trường `CHEXPERT_DATA_ROOT` tới thư mục có
`train.csv`, `valid.csv`, `train/`, `valid/`; checkpoint BCE được tìm trong `model/`.

## Đọc kết quả cuối

- `comparison_asl_vs_bce.csv`: từng nhãn và Macro, delta = ASL - BCE.
- `paired_patient_bootstrap.csv`: khoảng tin cậy 95% cho delta AUROC/AUPRC.
- `comparison_report.md`: nhận xét tự động, tập trung Pneumonia/Pneumothorax.
- Checkpoint ASL, history, thresholds, predictions và split manifest được lưu riêng.

Checkpoint được chọn bằng Macro AUROC trên **development**; mỗi model chọn
threshold riêng trên development với cùng mục tiêu recall >= 0.85.
Internal validation và official validation chỉ dùng để đánh giá.
Giá trị loss ASL/BCE khác thang đo, không so sánh trực tiếp training loss.

Checkpoint BCE cũ không có seed/split manifest đầy đủ: đánh giá cùng ảnh
cho biết chênh lệch giữa hai checkpoint, chưa đủ quy toàn bộ chênh lệch cho loss.
Bootstrap phản ánh biến thiên bệnh nhân của hai checkpoint cố định; cần nhiều
training seed để đánh giá độ ổn định của thay đổi loss.


In [ ]:
# 1. Imports + seed
import os
import json
import hashlib
from datetime import datetime, timezone
import random
from pathlib import Path
from copy import deepcopy

import numpy as np
import pandas as pd
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models

from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    precision_recall_curve,
)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

torch.backends.cudnn.deterministic = False
torch.backends.cudnn.benchmark = True

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('DEVICE =', DEVICE)

## 2. Cấu hình ASL

Chỉ thay loss nhị phân. Không fine-tune từ checkpoint BCE: DenseNet bắt đầu
lại từ ImageNet như lần train E2 gốc. Checkpoint BCE chỉ dùng để đánh giá đối chiếu.


In [ ]:
# 2. Config
DATA_ROOT = Path(os.environ.get('CHEXPERT_DATA_ROOT', '/kaggle/input/datasets/ashery/chexpert'))
TRAIN_CSV = DATA_ROOT / 'train.csv'
OFFICIAL_VALID_CSV = DATA_ROOT / 'valid.csv'

MAIN_LABELS = [
    'Atelectasis',
    'Cardiomegaly',
    'Edema',
    'Pleural Effusion',
    'Pneumonia',
    'Pneumothorax',
]

AUX_LABELS = [
    'Lung Opacity',
    'Consolidation',
    'Pleural Other',
]

# Relation targets chỉ dùng predicted probabilities của các source labels.
RELATION_SOURCES = {
    'Pneumonia': ['Lung Opacity', 'Consolidation', 'Atelectasis', 'Edema'],
    'Pneumothorax': ['Pleural Effusion', 'Pleural Other'],
}

EXPERIMENTS = {
    'B0_main6': {
        'use_aux': False,
        'use_relations': False,
        'pneumonia_policy': 'u_ignore',
        'pneumothorax_policy': 'u_ignore',
    },
    'E1_aux_multitask': {
        'use_aux': True,
        'use_relations': False,
        'pneumonia_policy': 'u_ignore',
        'pneumothorax_policy': 'u_ignore',
    },
    'E2_relation_aware': {
        'use_aux': True,
        'use_relations': True,
        'pneumonia_policy': 'u_ignore',
        'pneumothorax_policy': 'u_ignore',
    },
    'E3_relation_aware_lsr': {
        'use_aux': True,
        'use_relations': True,
        'pneumonia_policy': 'lsr_pos',
        'pneumothorax_policy': 'lsr_pos',
    },
}

EXPERIMENT_NAME = 'E2_relation_aware'
CFG = EXPERIMENTS[EXPERIMENT_NAME]

# Policy 4 nhãn chính giữ giống pipeline trước.
MAIN_POLICIES = {
    'Atelectasis': 'u_ones',
    'Cardiomegaly': 'u_multiclass',
    'Edema': 'u_ones',
    'Pleural Effusion': 'u_multiclass',
    'Pneumonia': CFG['pneumonia_policy'],
    'Pneumothorax': CFG['pneumothorax_policy'],
}

# Auxiliary labels: mặc định U-Ignore để không áp đặt hard mapping cho uncertain.
AUX_POLICIES = {
    'Lung Opacity': 'u_ignore',
    'Consolidation': 'u_ignore',
    'Pleural Other': 'u_ignore',
}

ACTIVE_LABELS = MAIN_LABELS + (AUX_LABELS if CFG['use_aux'] else [])
POLICIES = {**MAIN_POLICIES, **({k: AUX_POLICIES[k] for k in AUX_LABELS} if CFG['use_aux'] else {})}

# Dữ liệu / train
IMG_SIZE = 320
BATCH_SIZE = 16
NUM_WORKERS = 0 if os.name == 'nt' else 4
EPOCHS = 5
LR = 1e-4
WEIGHT_DECAY = 1e-5

# Missing labels: giữ 'zero' như pipeline cũ để E1/E2 chỉ thay auxiliary/relation.
# Sau khi có baseline sạch, thử 'ignore' như một ablation riêng cho cách xử lý NaN.
NAN_TRAIN_POLICY = 'zero'  # 'zero' để so sánh công bằng với pipeline cũ; thử 'ignore' như ablation riêng

# Loss
LOSS_MODE = 'asl'
POS_WEIGHT_CAP = 10.0
ASL_GAMMA_NEG = 4.0
ASL_GAMMA_POS = 1.0
ASL_CLIP = 0.05
AUX_LOSS_WEIGHT = 0.35

# Relation head
RELATION_DETACH = False  # False: gradient từ target có thể cập nhật source heads
RELATION_HIDDEN = 256

# Evaluation / model selection
TARGET_RECALL = 0.85
CHECKPOINT_METRIC = 'macro_auroc_main'  # ưu tiên AUROC vì mục tiêu là cải thiện AUROC
TRAIN_RATIO = 0.80
DEV_RATIO = 0.10
VALID_RATIO = 0.10

OUTPUT_ROOT = Path(os.environ.get('CHEXPERT_OUTPUT_ROOT', '/kaggle/working' if Path('/kaggle').exists() else 'outputs'))
RUN_ID = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
OUT_DIR = OUTPUT_ROOT / f'chexpert_e2_asl_{RUN_ID}'
OUT_DIR.mkdir(parents=True, exist_ok=False)
CHECKPOINT_PATH = OUT_DIR / f'best_{EXPERIMENT_NAME}_{LOSS_MODE}.pt'

print('Experiment:', EXPERIMENT_NAME)
print('Use aux:', CFG['use_aux'])
print('Use relations:', CFG['use_relations'])
print('Active labels:', ACTIVE_LABELS)
print('Policies:', POLICIES)
print('Checkpoint:', CHECKPOINT_PATH)

BOOTSTRAP_REPEATS = 300
BCE_FILENAME = 'best_E2_relation_aware_weighted_bce.pt'
BCE_CHECKPOINT_PATH = os.environ.get('BCE_CHECKPOINT_PATH')
if BCE_CHECKPOINT_PATH:
    BCE_CHECKPOINT_PATH = Path(BCE_CHECKPOINT_PATH)
elif Path('/kaggle/input').exists():
    candidates = sorted(Path('/kaggle/input').rglob(BCE_FILENAME))
    if len(candidates) != 1:
        raise FileNotFoundError(
            f'Tìm thấy {len(candidates)} checkpoint BCE. Gắn model/dataset chứa '
            f'{BCE_FILENAME} và đặt BCE_CHECKPOINT_PATH nếu có nhiều bản.'
        )
    BCE_CHECKPOINT_PATH = candidates[0]
else:
    candidates = [Path('model') / BCE_FILENAME, Path('..') / 'model' / BCE_FILENAME, Path(BCE_FILENAME)]
    BCE_CHECKPOINT_PATH = next((p for p in candidates if p.is_file()), candidates[0])

if not TRAIN_CSV.is_file():
    raise FileNotFoundError(f'Không có {TRAIN_CSV}. Gắn CheXpert hoặc đặt CHEXPERT_DATA_ROOT.')
if not BCE_CHECKPOINT_PATH.is_file():
    raise FileNotFoundError(f'Không có checkpoint BCE: {BCE_CHECKPOINT_PATH}')

# Chỉ nạp checkpoint của chính dự án hoặc nguồn đáng tin cậy.
bce_ckpt = torch.load(BCE_CHECKPOINT_PATH, map_location='cpu', weights_only=False)
expected_bce = {
    'active_labels': ACTIVE_LABELS,
    'policies': POLICIES,
    'experiment_name': EXPERIMENT_NAME,
    'use_relations': CFG['use_relations'],
    'relation_sources': RELATION_SOURCES,
    'loss_mode': 'weighted_bce',
    'nan_train_policy': NAN_TRAIN_POLICY,
}
for key, value in expected_bce.items():
    if bce_ckpt.get(key) != value:
        raise ValueError(f'Checkpoint BCE khác cấu hình {key}: {bce_ckpt.get(key)!r} != {value!r}')
print('BCE checkpoint:', BCE_CHECKPOINT_PATH)


## 3. Đọc dữ liệu và chia theo bệnh nhân

Chỉ giữ ảnh frontal và chia `train / development / internal validation` theo `PatientID` để tránh cùng bệnh nhân xuất hiện ở nhiều split.

Official validation **không được dùng để chọn checkpoint hoặc threshold**. Nếu có `valid.csv`, notebook chỉ đánh giá nó ở cuối sau khi đã khóa cấu hình.

In [ ]:
# 3. Load CSV + split patient-level
assert TRAIN_CSV.exists(), f'Không tìm thấy: {TRAIN_CSV}'

df = pd.read_csv(TRAIN_CSV)
required_cols = ['Path', 'Frontal/Lateral', *ACTIVE_LABELS]
missing_cols = [c for c in required_cols if c not in df.columns]
assert not missing_cols, f'Thiếu cột: {missing_cols}'

frontal_df = df[df['Frontal/Lateral'].astype(str).str.lower().eq('frontal')].copy()
frontal_df['PatientID'] = frontal_df['Path'].str.extract(r'(patient\d+)', expand=False)
assert frontal_df['PatientID'].notna().all(), 'Không tách được PatientID từ một số Path.'

patients = np.array(sorted(frontal_df['PatientID'].unique()))
rng = np.random.default_rng(SEED)
rng.shuffle(patients)

n_patients = len(patients)
n_train = int(n_patients * TRAIN_RATIO)
n_dev = int(n_patients * DEV_RATIO)

train_patients = set(patients[:n_train])
dev_patients = set(patients[n_train:n_train+n_dev])
valid_patients = set(patients[n_train+n_dev:])

assert train_patients.isdisjoint(dev_patients)
assert train_patients.isdisjoint(valid_patients)
assert dev_patients.isdisjoint(valid_patients)

train_df = frontal_df[frontal_df['PatientID'].isin(train_patients)].reset_index(drop=True)
dev_df = frontal_df[frontal_df['PatientID'].isin(dev_patients)].reset_index(drop=True)
valid_df = frontal_df[frontal_df['PatientID'].isin(valid_patients)].reset_index(drop=True)

split_summary = pd.DataFrame({
    'Split': ['Train', 'Develop', 'Internal valid'],
    'Images': [len(train_df), len(dev_df), len(valid_df)],
    'Patients': [
        train_df['PatientID'].nunique(),
        dev_df['PatientID'].nunique(),
        valid_df['PatientID'].nunique(),
    ],
})
print(split_summary.to_string(index=False))

split_frames = {'train': train_df, 'development': dev_df, 'internal_valid': valid_df}
split_hashes = {
    name: hashlib.sha256(frame[['Path', 'PatientID', *ACTIVE_LABELS]].to_csv(index=False).encode()).hexdigest()
    for name, frame in split_frames.items()
}
split_manifest = pd.concat([
    frame[['Path', 'PatientID']].assign(Split=name)
    for name, frame in split_frames.items()
], ignore_index=True)
split_manifest.to_csv(OUT_DIR / 'split_manifest.csv', index=False)
run_config = {
    'seed': SEED, 'train_csv_sha256': hashlib.sha256(TRAIN_CSV.read_bytes()).hexdigest(),
    'split_hashes': split_hashes, 'split_ratios': [TRAIN_RATIO, DEV_RATIO, VALID_RATIO],
    'img_size': IMG_SIZE, 'batch_size': BATCH_SIZE, 'epochs': EPOCHS,
    'optimizer': 'Adam', 'lr': LR, 'weight_decay': WEIGHT_DECAY,
    'aux_loss_weight': AUX_LOSS_WEIGHT, 'relation_hidden': RELATION_HIDDEN,
    'relation_detach': RELATION_DETACH, 'nan_train_policy': NAN_TRAIN_POLICY,
    'checkpoint_metric': CHECKPOINT_METRIC, 'target_recall': TARGET_RECALL,
    'loss_mode': LOSS_MODE, 'asl_gamma_neg': ASL_GAMMA_NEG,
    'asl_gamma_pos': ASL_GAMMA_POS, 'asl_clip': ASL_CLIP,
    'asl_focal_weight_grad': True, 'pos_weight_cap': POS_WEIGHT_CAP,
    'bce_checkpoint': str(BCE_CHECKPOINT_PATH),
    'bce_checkpoint_sha256': hashlib.sha256(BCE_CHECKPOINT_PATH.read_bytes()).hexdigest(),
    'source_notebook': 'CheXpert_relation_aware_Pneumonia_Pneumothorax.ipynb',
    'torch_version': torch.__version__, 'cuda_version': torch.version.cuda,
}
(OUT_DIR / 'run_config.json').write_text(json.dumps(run_config, indent=2), encoding='utf-8')


## 4. Kiểm tra phân bố nhãn và mối quan hệ trên train split

Cell dưới chỉ dùng các cặp nhãn raw `0/1` để tính thống kê quan hệ; `-1` và `NaN` bị bỏ khỏi phép tính. Mục đích là kiểm tra xem các source labels có thực sự cung cấp tín hiệu trên dữ liệu của bạn trước khi tin vào relation head.

In [ ]:
# 4. Raw label stats
def raw_label_stats(frame, labels):
    rows = []
    for label in labels:
        s = pd.to_numeric(frame[label], errors='coerce')
        rows.append({
            'Label': label,
            'Positive (1)': int((s == 1).sum()),
            'Negative (0)': int((s == 0).sum()),
            'Uncertain (-1)': int((s == -1).sum()),
            'NaN': int(s.isna().sum()),
        })
    return pd.DataFrame(rows)

stats_df = raw_label_stats(train_df, ACTIVE_LABELS)
stats_df

In [ ]:
# 5. Conditional prevalence cho các relation pairs
# Chỉ dùng rows mà cả target và source đều là 0/1.
def conditional_relation_table(frame, relation_sources):
    rows = []
    for target, sources in relation_sources.items():
        if target not in frame.columns:
            continue
        for source in sources:
            if source not in frame.columns:
                continue

            t = pd.to_numeric(frame[target], errors='coerce')
            s = pd.to_numeric(frame[source], errors='coerce')
            mask = t.isin([0, 1]) & s.isin([0, 1])
            tt = t[mask].astype(int)
            ss = s[mask].astype(int)

            n = int(mask.sum())
            n_s1 = int((ss == 1).sum())
            n_s0 = int((ss == 0).sum())
            p_t1_s1 = float(tt[ss == 1].mean()) if n_s1 else np.nan
            p_t1_s0 = float(tt[ss == 0].mean()) if n_s0 else np.nan

            rows.append({
                'Target': target,
                'Source': source,
                'N paired 0/1': n,
                'P(target=1 | source=1)': p_t1_s1,
                'P(target=1 | source=0)': p_t1_s0,
                'Risk ratio (descriptive)': (p_t1_s1 / p_t1_s0) if p_t1_s0 and not np.isnan(p_t1_s0) else np.nan,
            })
    return pd.DataFrame(rows)

relation_table = conditional_relation_table(train_df, RELATION_SOURCES)
relation_table.round(4)

## 5. Preprocessing

Giữ preprocessing gần pipeline trước để ablation tập trung vào thay đổi **auxiliary/relations**. Có thể bật augmentation nhẹ sau khi đã có baseline ổn định.

In [ ]:
# 6. Transforms
train_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    # Bật khi muốn làm ablation augmentation:
    # transforms.RandomAffine(degrees=5, translate=(0.02, 0.02), scale=(0.95, 1.05)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
])

eval_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
])

In [ ]:
# 7. Resolve image path
def resolve_image_path(path_str: str) -> Path:
    raw = str(path_str).replace('\\', '/')
    p = Path(raw)

    candidates = [DATA_ROOT / p, DATA_ROOT.parent / p]
    parts = list(p.parts)

    for prefix in ('CheXpert-v1.0-small', 'CheXpert-v1.0'):
        if prefix in parts:
            idx = parts.index(prefix)
            candidates.append(DATA_ROOT / Path(*parts[idx+1:]))

    for anchor in ('train', 'valid', 'test'):
        if anchor in parts:
            idx = parts.index(anchor)
            candidates.append(DATA_ROOT / Path(*parts[idx:]))

    for c in candidates:
        if c.exists():
            return c

    raise FileNotFoundError(
        f'Không tìm thấy ảnh. CSV Path: {path_str}\n'
        f'DATA_ROOT: {DATA_ROOT}\nĐã thử:\n' + '\n'.join(str(c) for c in candidates)
    )

example_path = resolve_image_path(frontal_df.iloc[0]['Path'])
print('Example:', example_path)
print('Exists:', example_path.exists())

## 6. Dataset và uncertainty policies

- `u_ones`: `-1 -> 1`
- `u_zeros`: `-1 -> 0`
- `u_ignore`: bỏ `-1` khỏi loss
- `u_multiclass`: `0 / 1 / uncertain` là 3 lớp
- `lsr_pos`: `-1 -> Uniform(0.55, 0.85)`
- `lsr_neg`: `-1 -> Uniform(0.00, 0.30)`

Để so sánh công bằng với pipeline cũ, notebook mặc định `NaN -> 0` khi training (`NAN_TRAIN_POLICY='zero'`). Sau khi hoàn tất B0/E1/E2/E3, có thể chạy thêm một ablation riêng với `NAN_TRAIN_POLICY='ignore'`.

In [ ]:
# 8. Dataset
class CheXpertRelationDataset(Dataset):
    def __init__(self, frame, labels, policies, transform=None, training=False, nan_train_policy='ignore'):
        self.frame = frame.reset_index(drop=True)
        self.labels = labels
        self.policies = policies
        self.transform = transform
        self.training = training
        self.nan_train_policy = nan_train_policy

    def __len__(self):
        return len(self.frame)

    def _binary_target(self, raw, policy):
        if pd.isna(raw):
            if self.nan_train_policy == 'zero':
                return 0.0, 1.0
            return 0.0, 0.0

        raw = float(raw)
        if raw == 1.0:
            return 1.0, 1.0
        if raw == 0.0:
            return 0.0, 1.0
        if raw == -1.0:
            if policy == 'u_ones':
                return 1.0, 1.0
            if policy == 'u_zeros':
                return 0.0, 1.0
            if policy == 'u_ignore':
                return 0.0, 0.0
            if policy == 'lsr_pos':
                return float(np.random.uniform(0.55, 0.85)), 1.0
            if policy == 'lsr_neg':
                return float(np.random.uniform(0.00, 0.30)), 1.0
        raise ValueError(f'Không hỗ trợ raw={raw}, policy={policy}')

    def __getitem__(self, idx):
        row = self.frame.iloc[idx]
        img_path = resolve_image_path(row['Path'])
        image = Image.open(img_path).convert('RGB')
        if self.transform is not None:
            image = self.transform(image)

        binary_targets = {}
        binary_masks = {}
        multiclass_targets = {}
        multiclass_masks = {}
        raw_values = {}
        eval_masks = {}

        for label in self.labels:
            raw = pd.to_numeric(pd.Series([row[label]]), errors='coerce').iloc[0]
            raw_values[label] = np.nan if pd.isna(raw) else float(raw)
            eval_masks[label] = float((not pd.isna(raw)) and float(raw) in (0.0, 1.0))

            policy = self.policies[label]
            if policy == 'u_multiclass':
                if pd.isna(raw):
                    target = 0
                    m = 1.0 if self.nan_train_policy == 'zero' else 0.0
                elif float(raw) == 0.0:
                    target, m = 0, 1.0
                elif float(raw) == 1.0:
                    target, m = 1, 1.0
                elif float(raw) == -1.0:
                    target, m = 2, 1.0
                else:
                    raise ValueError(raw)
                multiclass_targets[label] = target
                multiclass_masks[label] = m
            else:
                target, m = self._binary_target(raw, policy)
                binary_targets[label] = target
                binary_masks[label] = m

        return {
            'image': image,
            'path': str(row['Path']),
            'patient_id': row.get('PatientID', ''),
            'binary_targets': {k: torch.tensor(v, dtype=torch.float32) for k, v in binary_targets.items()},
            'binary_masks': {k: torch.tensor(v, dtype=torch.float32) for k, v in binary_masks.items()},
            'multiclass_targets': {k: torch.tensor(v, dtype=torch.long) for k, v in multiclass_targets.items()},
            'multiclass_masks': {k: torch.tensor(v, dtype=torch.float32) for k, v in multiclass_masks.items()},
            'raw_values': {k: torch.tensor(0.0 if np.isnan(v) else v, dtype=torch.float32) for k, v in raw_values.items()},
            'eval_masks': {k: torch.tensor(v, dtype=torch.float32) for k, v in eval_masks.items()},
        }

In [ ]:
# 9. DataLoaders
train_ds = CheXpertRelationDataset(
    train_df, ACTIVE_LABELS, POLICIES, train_transform,
    training=True, nan_train_policy=NAN_TRAIN_POLICY
)
dev_ds = CheXpertRelationDataset(
    dev_df, ACTIVE_LABELS, POLICIES, eval_transform,
    training=False, nan_train_policy=NAN_TRAIN_POLICY
)
valid_ds = CheXpertRelationDataset(
    valid_df, ACTIVE_LABELS, POLICIES, eval_transform,
    training=False, nan_train_policy=NAN_TRAIN_POLICY
)

train_loader = DataLoader(
    train_ds, batch_size=BATCH_SIZE, shuffle=True,
    num_workers=NUM_WORKERS, pin_memory=True,
)
dev_loader = DataLoader(
    dev_ds, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=NUM_WORKERS, pin_memory=True,
)
valid_loader = DataLoader(
    valid_ds, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=NUM_WORKERS, pin_memory=True,
)

sample = train_ds[0]
print('image:', sample['image'].shape)
print('binary labels:', list(sample['binary_targets']))
print('multiclass labels:', list(sample['multiclass_targets']))

## 7. Pos-weight

Với `NAN_TRAIN_POLICY='zero'`, NaN được tính như negative để giữ đúng baseline cũ. Nếu đổi sang `'ignore'`, NaN sẽ bị loại khỏi cả loss và phép tính `pos_weight`. Với `u_ignore`, uncertain luôn bị loại khỏi loss.

In [ ]:
# 10. Binary/multiclass labels + pos weights
BINARY_LABELS = [l for l in ACTIVE_LABELS if POLICIES[l] != 'u_multiclass']
MULTICLASS_LABELS = [l for l in ACTIVE_LABELS if POLICIES[l] == 'u_multiclass']

def compute_binary_pos_weights(frame, labels, policies, nan_train_policy='ignore'):
    result = {}
    rows = []

    for label in labels:
        if policies[label] == 'u_multiclass':
            continue

        policy = policies[label]
        s = pd.to_numeric(frame[label], errors='coerce')

        if policy == 'u_ones':
            pos = int(((s == 1) | (s == -1)).sum())
            neg = int((s == 0).sum())
        elif policy == 'u_zeros':
            pos = int((s == 1).sum())
            neg = int(((s == 0) | (s == -1)).sum())
        elif policy == 'u_ignore':
            pos = int((s == 1).sum())
            neg = int((s == 0).sum())
        elif policy in ('lsr_pos', 'lsr_neg'):
            # Pos-weight dựa trên explicit 0/1; uncertain có soft target riêng trong loss.
            pos = int((s == 1).sum())
            neg = int((s == 0).sum())
        else:
            raise ValueError(policy)

        if nan_train_policy == 'zero':
            neg += int(s.isna().sum())

        raw_w = neg / max(pos, 1)
        used_w = min(raw_w, POS_WEIGHT_CAP)
        result[label] = used_w
        rows.append({
            'Label': label,
            'Policy': policy,
            'Pos count': pos,
            'Neg count': neg,
            'Raw pos_weight': raw_w,
            'Used pos_weight': used_w,
        })

    return result, pd.DataFrame(rows)

POS_WEIGHTS, pos_weight_table = compute_binary_pos_weights(
    train_df, ACTIVE_LABELS, POLICIES, NAN_TRAIN_POLICY
)
pos_weight_table

## 8. Relation-aware DenseNet121

Cách hoạt động:

1. DenseNet121 tạo shared image feature.
2. Mỗi active label có một `base head`.
3. Nếu bật relation learning:
   - `Pneumonia` lấy thêm predicted probabilities của `Lung Opacity`, `Consolidation`, `Atelectasis`, `Edema`.
   - `Pneumothorax` lấy thêm predicted probabilities của `Pleural Effusion`, `Pleural Other`.
4. Relation head dự đoán một **delta logit** và cộng vào base logit của target.

Vì source signals là **dự đoán của mô hình**, không phải ground-truth, cấu trúc này dùng được khi inference trên ảnh mới.

In [ ]:
# 11. Helpers chuyển logits -> positive probability
def positive_probability_from_logits(label, logits, policies):
    if policies[label] == 'u_multiclass':
        # Restricted softmax trên negative(0) và positive(1), bỏ uncertain(2)
        return torch.softmax(logits[:, :2], dim=1)[:, 1]
    return torch.sigmoid(logits.squeeze(1))


class RelationAwareDenseNet121(nn.Module):
    def __init__(
        self,
        active_labels,
        policies,
        relation_sources,
        use_relations=True,
        relation_hidden=256,
        relation_detach=False,
        pretrained=True,
    ):
        super().__init__()
        self.active_labels = list(active_labels)
        self.policies = policies
        self.use_relations = use_relations
        self.relation_detach = relation_detach

        weights = models.DenseNet121_Weights.IMAGENET1K_V1 if pretrained else None
        net = models.densenet121(weights=weights)
        in_features = net.classifier.in_features
        net.classifier = nn.Identity()
        self.backbone = net
        self.in_features = in_features

        self.base_heads = nn.ModuleDict()
        for label in self.active_labels:
            out_dim = 3 if policies[label] == 'u_multiclass' else 1
            self.base_heads[label] = nn.Linear(in_features, out_dim)

        # Chỉ tạo relation head khi target + source labels đều active.
        self.relation_sources = {}
        self.relation_heads = nn.ModuleDict()

        if self.use_relations:
            for target, sources in relation_sources.items():
                if target not in self.active_labels:
                    continue
                assert policies[target] != 'u_multiclass', f'{target} relation target phải là binary head.'
                valid_sources = [s for s in sources if s in self.active_labels]
                if not valid_sources:
                    continue
                self.relation_sources[target] = valid_sources

                # feature + base target logit + one probability per source
                rel_dim = in_features + 1 + len(valid_sources)
                self.relation_heads[target] = nn.Sequential(
                    nn.Linear(rel_dim, relation_hidden),
                    nn.ReLU(inplace=True),
                    nn.Dropout(p=0.2),
                    nn.Linear(relation_hidden, 1),
                )

    def forward(self, x):
        feat = self.backbone(x)
        base = {label: self.base_heads[label](feat) for label in self.active_labels}
        outputs = dict(base)

        if not self.use_relations:
            return outputs

        for target, sources in self.relation_sources.items():
            source_probs = []
            for source in sources:
                p = positive_probability_from_logits(source, base[source], self.policies)
                if self.relation_detach:
                    p = p.detach()
                source_probs.append(p.unsqueeze(1))

            base_target_logit = base[target].squeeze(1)
            relation_input = torch.cat(
                [feat, base_target_logit.unsqueeze(1), *source_probs], dim=1
            )
            delta = self.relation_heads[target](relation_input).squeeze(1)
            final_logit = base_target_logit + delta
            outputs[target] = final_logit.unsqueeze(1)

        return outputs

model = RelationAwareDenseNet121(
    active_labels=ACTIVE_LABELS,
    policies=POLICIES,
    relation_sources=RELATION_SOURCES,
    use_relations=CFG['use_relations'],
    relation_hidden=RELATION_HIDDEN,
    relation_detach=RELATION_DETACH,
    pretrained=True,
).to(DEVICE)

print('Trainable parameters:', sum(p.numel() for p in model.parameters() if p.requires_grad))
print('Relation sources used:', model.relation_sources)

## 9. Loss: ASL cho binary heads, CE cho multiclass heads

Main tasks có trọng số 1.0, auxiliary tasks 0.35 như E2 gốc.
Mask uncertain/NaN được giữ nguyên. ASL dùng mean trên các phần tử hợp lệ
của mỗi task như nhánh BCE; không dùng tổng loss chưa chuẩn hóa từ implementation tham khảo.


In [ ]:
# 12. Asymmetric Loss
class AsymmetricLoss(nn.Module):
    def __init__(self, gamma_neg=4.0, gamma_pos=1.0, clip=0.05, eps=1e-8):
        super().__init__()
        self.gamma_neg = gamma_neg
        self.gamma_pos = gamma_pos
        self.clip = clip
        self.eps = eps

    def forward(self, logits, targets):
        x_sigmoid = torch.sigmoid(logits)
        xs_pos = x_sigmoid
        xs_neg = 1.0 - x_sigmoid

        if self.clip is not None and self.clip > 0:
            xs_neg = (xs_neg + self.clip).clamp(max=1.0)

        loss = targets * torch.log(xs_pos.clamp(min=self.eps))
        loss += (1.0 - targets) * torch.log(xs_neg.clamp(min=self.eps))

        if self.gamma_neg > 0 or self.gamma_pos > 0:
            pt = xs_pos * targets + xs_neg * (1.0 - targets)
            gamma = self.gamma_pos * targets + self.gamma_neg * (1.0 - targets)
            loss *= torch.pow(1.0 - pt, gamma)

        return -loss

asl_loss = AsymmetricLoss(ASL_GAMMA_NEG, ASL_GAMMA_POS, ASL_CLIP)

In [ ]:
# 13. Tổng loss
def label_task_weight(label):
    return 1.0 if label in MAIN_LABELS else AUX_LOSS_WEIGHT


def compute_loss(outputs, batch):
    total = torch.tensor(0.0, device=DEVICE)
    total_weight = 0.0
    parts = {}

    # Binary heads
    for label in BINARY_LABELS:
        logits = outputs[label].squeeze(1)
        targets = batch['binary_targets'][label].to(DEVICE)
        mask = batch['binary_masks'][label].to(DEVICE)

        if LOSS_MODE == 'weighted_bce':
            pos_weight = torch.tensor(POS_WEIGHTS[label], dtype=torch.float32, device=DEVICE)
            per_item = F.binary_cross_entropy_with_logits(
                logits, targets, pos_weight=pos_weight, reduction='none'
            )
        elif LOSS_MODE == 'asl':
            per_item = asl_loss(logits, targets)
        else:
            raise ValueError(LOSS_MODE)

        denom = mask.sum().clamp(min=1.0)
        label_loss = (per_item * mask).sum() / denom
        w = label_task_weight(label)
        total = total + w * label_loss
        total_weight += w
        parts[label] = float(label_loss.detach().cpu())

    # Multiclass heads
    for label in MULTICLASS_LABELS:
        logits = outputs[label]
        targets = batch['multiclass_targets'][label].to(DEVICE)
        mask = batch['multiclass_masks'][label].to(DEVICE)

        per_item = F.cross_entropy(logits, targets, reduction='none')
        denom = mask.sum().clamp(min=1.0)
        label_loss = (per_item * mask).sum() / denom
        w = label_task_weight(label)
        total = total + w * label_loss
        total_weight += w
        parts[label] = float(label_loss.detach().cpu())

    total = total / max(total_weight, 1e-8)
    return total, parts

## 10. Prediction + ranking metrics

Checkpoint được chọn theo **Macro AUROC của 6 MAIN_LABELS trên development set**. Auxiliary labels không tham gia tiêu chí chọn checkpoint.

AUROC/AUPRC chỉ được tính trên raw labels `0/1`; `-1` và `NaN` không tham gia metric.

In [ ]:
# 14. Collect predictions + metrics
@torch.no_grad()
def collect_predictions(model, loader, labels=None):
    model.eval()
    labels = labels or ACTIVE_LABELS
    store = {label: {'y': [], 'p': [], 'mask': []} for label in labels}
    paths, patient_ids = [], []
    for batch in loader:
        images = batch['image'].to(DEVICE, non_blocking=True)
        outputs = model(images)
        paths.extend(batch['path'])
        patient_ids.extend(batch['patient_id'])
        for label in labels:
            store[label]['p'].append(positive_probability_from_logits(label, outputs[label], POLICIES).cpu().numpy())
            store[label]['y'].append(batch['raw_values'][label].numpy())
            store[label]['mask'].append(batch['eval_masks'][label].numpy().astype(bool))
    result = {}
    for label in labels:
        y = np.concatenate(store[label]['y'])
        p = np.concatenate(store[label]['p'])
        mask = np.concatenate(store[label]['mask'])
        result[label] = {
            'y': y[mask].astype(int), 'p': p[mask],
            'path': np.asarray(paths)[mask], 'patient_id': np.asarray(patient_ids)[mask],
        }
    return result


def safe_auroc(y, p):
    return np.nan if len(np.unique(y)) < 2 else roc_auc_score(y, p)


def safe_auprc(y, p):
    return np.nan if len(np.unique(y)) < 2 else average_precision_score(y, p)


def ranking_metrics(pred_dict, labels):
    rows = []
    for label in labels:
        y, p = pred_dict[label]['y'], pred_dict[label]['p']
        rows.append({
            'Label': label, 'N_eval': len(y), 'Positive': int(y.sum()),
            'Negative': int((1-y).sum()), 'AUROC': safe_auroc(y, p), 'AUPRC': safe_auprc(y, p),
        })
    return pd.DataFrame(rows)


In [ ]:
# 15. Train
optimizer = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)

history = []
best_score = -np.inf
best_state = None

for epoch in range(1, EPOCHS + 1):
    model.train()
    running = 0.0

    for step, batch in enumerate(train_loader, start=1):
        images = batch['image'].to(DEVICE, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)

        outputs = model(images)
        loss, parts = compute_loss(outputs, batch)
        if not torch.isfinite(loss):
            raise FloatingPointError(f'Non-finite {LOSS_MODE} loss at epoch {epoch}, step {step}')
        loss.backward()
        optimizer.step()

        running += float(loss.detach().cpu())
        if step % 100 == 0:
            print(f'Epoch {epoch} | step {step}/{len(train_loader)} | loss={running/step:.4f}')

    train_loss = running / max(len(train_loader), 1)

    dev_pred = collect_predictions(model, dev_loader, MAIN_LABELS)
    dev_metrics = ranking_metrics(dev_pred, MAIN_LABELS)
    macro_auroc = dev_metrics['AUROC'].mean()
    macro_auprc = dev_metrics['AUPRC'].mean()

    if CHECKPOINT_METRIC == 'macro_auroc_main':
        score = macro_auroc
    elif CHECKPOINT_METRIC == 'macro_auprc_main':
        score = macro_auprc
    else:
        raise ValueError(CHECKPOINT_METRIC)

    row = {
        'Epoch': epoch,
        'TrainLoss': train_loss,
        'MacroAUROC_MAIN': macro_auroc,
        'MacroAUPRC_MAIN': macro_auprc,
        'CheckpointScore': score,
    }
    history.append(row)
    pd.DataFrame(history).to_csv(OUT_DIR / f'{EXPERIMENT_NAME}_{LOSS_MODE}_history.csv', index=False)

    print('\n', pd.DataFrame([row]).round(5).to_string(index=False))
    print(dev_metrics[['Label', 'AUROC', 'AUPRC']].round(4).to_string(index=False))

    if score > best_score:
        best_score = score
        best_state = deepcopy(model.state_dict())
        torch.save({
            'model_state_dict': best_state,
            'run_config': run_config,
            'active_labels': ACTIVE_LABELS,
            'policies': POLICIES,
            'experiment_name': EXPERIMENT_NAME,
            'use_relations': CFG['use_relations'],
            'relation_sources': model.relation_sources,
            'loss_mode': LOSS_MODE,
            'pos_weights': POS_WEIGHTS,
            'nan_train_policy': NAN_TRAIN_POLICY,
            'epoch': epoch,
            'dev_macro_auroc': macro_auroc,
            'dev_macro_auprc': macro_auprc,
        }, CHECKPOINT_PATH)
        print('Saved best checkpoint ->', CHECKPOINT_PATH)

history_df = pd.DataFrame(history)
history_df
history_df.to_csv(OUT_DIR / f'{EXPERIMENT_NAME}_{LOSS_MODE}_history.csv', index=False)


In [ ]:
# 16. Load best checkpoint
ckpt = torch.load(CHECKPOINT_PATH, map_location=DEVICE, weights_only=False)
model.load_state_dict(ckpt['model_state_dict'])
print('Best epoch:', ckpt['epoch'])
print('Best dev Macro AUROC:', ckpt['dev_macro_auroc'])
print('Best dev Macro AUPRC:', ckpt['dev_macro_auprc'])

## 11. Chọn threshold trên development set

Threshold không ảnh hưởng AUROC, nhưng cần cho Precision/Recall/Specificity/F1. Notebook giữ mục tiêu `Recall >= 0.85`, sau đó chọn threshold có precision cao nhất trong các candidate đạt recall mục tiêu.

In [ ]:
# 17. Chọn threshold high-recall trên development set
def choose_threshold_for_recall(y_true, y_prob, target_recall=0.85):
    if len(np.unique(y_true)) < 2:
        raise ValueError('Development cần có cả positive và negative để chọn threshold.')
    precision, recall, thresholds = precision_recall_curve(y_true, y_prob)
    p = precision[:-1]
    r = recall[:-1]

    candidates = np.where(r >= target_recall)[0]
    if len(candidates) == 0:
        idx = int(np.argmax(r))
    else:
        best_local = int(np.argmax(p[candidates]))
        idx = int(candidates[best_local])

    return {
        'threshold': float(thresholds[idx]),
        'precision_dev': float(p[idx]),
        'recall_dev': float(r[idx]),
    }

best_dev_pred = collect_predictions(model, dev_loader, MAIN_LABELS)
thresholds = {}
threshold_rows = []

for label in MAIN_LABELS:
    y = best_dev_pred[label]['y']
    p = best_dev_pred[label]['p']
    info = choose_threshold_for_recall(y, p, TARGET_RECALL)
    thresholds[label] = info['threshold']
    threshold_rows.append({'Label': label, **info})

threshold_table = pd.DataFrame(threshold_rows)
threshold_table

In [ ]:
# 18. Threshold-based metrics
def threshold_metrics(pred_dict, thresholds, labels):
    rows = []
    for label in labels:
        y = pred_dict[label]['y']
        p = pred_dict[label]['p']
        t = thresholds[label]
        yhat = (p >= t).astype(int)

        tn, fp, fn, tp = confusion_matrix(y, yhat, labels=[0, 1]).ravel()
        specificity = tn / (tn + fp) if (tn + fp) else np.nan

        rows.append({
            'Label': label, 'N_eval': len(y), 'Positive': int(y.sum()), 'Negative': int((1-y).sum()),
            'Threshold': t,
            'AUROC': safe_auroc(y, p),
            'AUPRC': safe_auprc(y, p),
            'Precision': precision_score(y, yhat, zero_division=0),
            'Recall': recall_score(y, yhat, zero_division=0),
            'Specificity': specificity,
            'F1': f1_score(y, yhat, zero_division=0),
            'TP': int(tp),
            'FP': int(fp),
            'TN': int(tn),
            'FN': int(fn),
        })
    return pd.DataFrame(rows)

valid_pred = collect_predictions(model, valid_loader, MAIN_LABELS)
valid_results = threshold_metrics(valid_pred, thresholds, MAIN_LABELS)
valid_results.round(4)

In [ ]:
# 19. Macro summary + tập trung 2 nhãn cần cải thiện
macro_cols = ['AUROC', 'AUPRC', 'Precision', 'Recall', 'Specificity', 'F1']
macro_summary = valid_results[macro_cols].mean().to_frame('Macro').T
focus_results = valid_results[valid_results['Label'].isin(['Pneumonia', 'Pneumothorax'])].copy()

print('Macro summary')
print(macro_summary.round(4).to_string())
print('\nFocus labels')
print(focus_results.round(4).to_string(index=False))

## 12. Optional: đánh giá official validation sau khi đã khóa cấu hình

Cell này chỉ chạy nếu tìm thấy `valid.csv`. Không dùng official validation để chọn policy, checkpoint hoặc threshold.

In [ ]:
# 20. Official validation (optional): không chọn checkpoint/threshold tại đây.
official_pred = None
if OFFICIAL_VALID_CSV.is_file():
    official_valid_df = pd.read_csv(OFFICIAL_VALID_CSV)
    required = ['Path', *MAIN_LABELS]
    missing = [c for c in required if c not in official_valid_df.columns]
    if missing:
        raise ValueError(f'Official valid thiếu cột: {missing}')
    official_valid_df['PatientID'] = official_valid_df['Path'].str.extract(r'(patient\d+)', expand=False)
    if official_valid_df['PatientID'].isna().any():
        raise ValueError('Không tách được PatientID của official validation.')
    if set(official_valid_df['PatientID']) & set(frontal_df['PatientID']):
        raise ValueError('Official validation trùng bệnh nhân với train.csv.')
    official_ds = CheXpertRelationDataset(
        official_valid_df, MAIN_LABELS, POLICIES, eval_transform,
        training=False, nan_train_policy=NAN_TRAIN_POLICY,
    )
    official_loader = DataLoader(
        official_ds, batch_size=BATCH_SIZE, shuffle=False,
        num_workers=NUM_WORKERS, pin_memory=True,
    )
    official_pred = collect_predictions(model, official_loader, MAIN_LABELS)
    official_results = threshold_metrics(official_pred, thresholds, MAIN_LABELS)
    official_results.to_csv(OUT_DIR / f'{EXPERIMENT_NAME}_{LOSS_MODE}_official_valid.csv', index=False)
    print(official_results.round(4).to_string(index=False))
else:
    print('Không có official valid.csv; chỉ báo cáo internal validation.')


## 13. Lưu kết quả

Mỗi experiment tạo file riêng để sau đó ghép bảng ablation.

In [ ]:
# 21. Save results
history_df.to_csv(OUT_DIR / f'{EXPERIMENT_NAME}_{LOSS_MODE}_history.csv', index=False)
threshold_table.to_csv(OUT_DIR / f'{EXPERIMENT_NAME}_{LOSS_MODE}_thresholds.csv', index=False)
valid_results.to_csv(OUT_DIR / f'{EXPERIMENT_NAME}_{LOSS_MODE}_internal_valid.csv', index=False)
macro_summary.to_csv(OUT_DIR / f'{EXPERIMENT_NAME}_{LOSS_MODE}_macro.csv', index=False)
relation_table.to_csv(OUT_DIR / f'{EXPERIMENT_NAME}_relation_stats.csv', index=False)

print('Saved to:', OUT_DIR.resolve())

## 14. So sánh trực tiếp ASL với checkpoint weighted BCE

Nạp BCE với đúng kiến trúc, đánh giá lại development để chọn threshold cho BCE,
rồi đánh giá trên cùng internal/official validation. Delta dương nghĩa là ASL cao hơn
với metric đó; cần xem đồng thời precision và recall để hiểu trade-off.

Bootstrap lấy mẫu **bệnh nhân**, dùng cùng mẫu cho hai checkpoint. CI chỉ nói về
hai checkpoint cố định, không đo biến thiên khi train lại. Không tune ASL dựa vào
internal/official validation sau khi xem kết quả.


In [ ]:
# 22. Ghép kết quả và paired patient bootstrap
METRIC_COLUMNS = ['AUROC', 'AUPRC', 'Precision', 'Recall', 'Specificity', 'F1']


def compare_metrics(asl_results, bce_results):
    a = asl_results.set_index('Label')
    b = bce_results.set_index('Label').reindex(a.index)
    if not a[['N_eval', 'Positive', 'Negative']].equals(b[['N_eval', 'Positive', 'Negative']]):
        raise ValueError('ASL/BCE khác số mẫu hoặc ground truth.')
    result = a.join(b, lsuffix='_ASL', rsuffix='_BCE')
    for metric in METRIC_COLUMNS:
        result.loc['Macro', f'{metric}_ASL'] = a[metric].mean()
        result.loc['Macro', f'{metric}_BCE'] = b[metric].mean()
        result[f'Delta_{metric}'] = result[f'{metric}_ASL'] - result[f'{metric}_BCE']
    return result.rename_axis('Label').reset_index()


def paired_patient_bootstrap(asl_pred, bce_pred, labels, repeats=300, seed=42):
    patients = np.unique(np.concatenate([asl_pred[label]['patient_id'] for label in labels]))
    if not len(patients):
        raise ValueError('Không có bệnh nhân để bootstrap.')
    positions = {}
    for label in labels:
        for key in ('path', 'patient_id', 'y'):
            if not np.array_equal(asl_pred[label][key], bce_pred[label][key]):
                raise ValueError(f'ASL/BCE không cùng mẫu {label}/{key}.')
        positions[label] = np.searchsorted(patients, asl_pred[label]['patient_id'])
    metric_functions = {'AUROC': roc_auc_score, 'AUPRC': average_precision_score}
    draws = {(label, metric): [] for label in [*labels, 'Macro'] for metric in metric_functions}
    rng = np.random.default_rng(seed)
    for _ in range(repeats):
        counts = np.bincount(rng.integers(len(patients), size=len(patients)), minlength=len(patients))
        current = {metric: [] for metric in metric_functions}
        for label in labels:
            y = asl_pred[label]['y']
            w = counts[positions[label]]
            if len(np.unique(y[w > 0])) < 2:
                continue
            for metric, fn in metric_functions.items():
                delta = fn(y, asl_pred[label]['p'], sample_weight=w) - fn(y, bce_pred[label]['p'], sample_weight=w)
                draws[label, metric].append(delta)
                current[metric].append(delta)
        for metric, values in current.items():
            if len(values) == len(labels):
                draws['Macro', metric].append(float(np.mean(values)))
    rows = []
    for (label, metric), values in draws.items():
        low, high = np.quantile(values, [0.025, 0.975]) if values else (np.nan, np.nan)
        rows.append({'Label': label, 'Metric': metric, 'CI95_low': low, 'CI95_high': high, 'N_bootstrap': len(values)})
    return pd.DataFrame(rows)


In [ ]:
# 23. Đánh giá checkpoint BCE trên cùng cohort
model.to('cpu')
if torch.cuda.is_available():
    torch.cuda.empty_cache()
bce_model = RelationAwareDenseNet121(
    active_labels=ACTIVE_LABELS, policies=POLICIES, relation_sources=RELATION_SOURCES,
    use_relations=CFG['use_relations'], relation_hidden=RELATION_HIDDEN,
    relation_detach=RELATION_DETACH, pretrained=False,
).to(DEVICE)
bce_model.load_state_dict(bce_ckpt['model_state_dict'], strict=True)
bce_dev_pred = collect_predictions(bce_model, dev_loader, MAIN_LABELS)
bce_threshold_rows = []
for label in MAIN_LABELS:
    info = choose_threshold_for_recall(bce_dev_pred[label]['y'], bce_dev_pred[label]['p'], TARGET_RECALL)
    bce_threshold_rows.append({'Label': label, **info})
bce_threshold_table = pd.DataFrame(bce_threshold_rows)
bce_thresholds = dict(zip(bce_threshold_table['Label'], bce_threshold_table['threshold']))
bce_threshold_table.to_csv(OUT_DIR / 'E2_relation_aware_weighted_bce_thresholds.csv', index=False)

evaluation_sets = {'internal_valid': (valid_loader, valid_pred, valid_results)}
if official_pred is not None:
    evaluation_sets['official_valid'] = (official_loader, official_pred, official_results)
comparison_tables, bootstrap_tables = [], []
for split, (loader, asl_pred, asl_results) in evaluation_sets.items():
    bce_pred = collect_predictions(bce_model, loader, MAIN_LABELS)
    bce_results = threshold_metrics(bce_pred, bce_thresholds, MAIN_LABELS)
    # Bootstrap cũng kiểm tra Path/PatientID/ground truth giống nhau trước khi so sánh.
    ci = paired_patient_bootstrap(asl_pred, bce_pred, MAIN_LABELS, BOOTSTRAP_REPEATS, SEED)
    bootstrap_tables.append(ci.assign(Split=split))
    comparison = compare_metrics(asl_results, bce_results).assign(Split=split)
    comparison_tables.append(comparison)
    bce_results.to_csv(OUT_DIR / f'E2_relation_aware_weighted_bce_{split}.csv', index=False)
    paired_predictions = pd.concat([
        pd.DataFrame({
            'Path': asl_pred[label]['path'], 'PatientID': asl_pred[label]['patient_id'],
            'Label': label, 'y': asl_pred[label]['y'],
            'p_ASL': asl_pred[label]['p'], 'p_BCE': bce_pred[label]['p'],
        }) for label in MAIN_LABELS
    ], ignore_index=True)
    paired_predictions.to_csv(OUT_DIR / f'{split}_paired_predictions.csv.gz', index=False)
    print('\n', split)
    print(comparison[['Label', 'AUROC_BCE', 'AUROC_ASL', 'Delta_AUROC', 'AUPRC_BCE', 'AUPRC_ASL', 'Delta_AUPRC']].round(5).to_string(index=False))

comparison_df = pd.concat(comparison_tables, ignore_index=True)
bootstrap_df = pd.concat(bootstrap_tables, ignore_index=True)
comparison_df.to_csv(OUT_DIR / 'comparison_asl_vs_bce.csv', index=False)
bootstrap_df.to_csv(OUT_DIR / 'paired_patient_bootstrap.csv', index=False)


In [ ]:
# 24. Nhận xét tự động và báo cáo Markdown
report = [
    '# DenseNet E2: ASL so với weighted BCE', '',
    f'- ASL checkpoint: `{CHECKPOINT_PATH.name}`, best epoch {ckpt["epoch"]}.',
    f'- BCE checkpoint: `{BCE_CHECKPOINT_PATH}`, best epoch {bce_ckpt.get("epoch", "unknown")}.',
    '- Delta = ASL - BCE; +0.01 tương ứng +1 điểm phần trăm.',
    '- Checkpoint/threshold chỉ chọn trên development; validation chỉ dùng để báo cáo.',
    '- Hai head u_multiclass vẫn dùng CE; ASL thay weighted BCE ở binary heads.', '',
]
bce_config = bce_ckpt.get('run_config', {})
control_keys = [
    'seed', 'train_csv_sha256', 'split_hashes', 'split_ratios', 'img_size', 'batch_size',
    'epochs', 'optimizer', 'lr', 'weight_decay', 'aux_loss_weight',
    'relation_hidden', 'relation_detach', 'nan_train_policy', 'checkpoint_metric', 'target_recall',
]
unknown = [key for key in control_keys if key not in bce_config]
mismatches = [key for key in control_keys if key in bce_config and bce_config[key] != run_config[key]]
if unknown or mismatches:
    report += ['## Giới hạn đối chứng', '']
    if unknown:
        report += [f'BCE thiếu metadata: {", ".join(unknown)}.']
    if mismatches:
        report += [f'Cấu hình khác nhau: {", ".join(mismatches)}.']
    report += [
        'Đây là so sánh hai checkpoint trên cùng ảnh; chưa thể quy toàn bộ chênh lệch cho loss.',
        'Muốn ablation có kiểm soát, train lại BCE với cùng dữ liệu, split và cấu hình đã lưu.', '',
    ]
for split in comparison_df['Split'].unique():
    report += [f'## {split}', '', '| Label | Metric | BCE | ASL | Delta | 95% CI của delta | Nhận xét |',
               '|---|---|---:|---:|---:|---|---|']
    table = comparison_df[comparison_df['Split'].eq(split)].set_index('Label')
    intervals = bootstrap_df[bootstrap_df['Split'].eq(split)].set_index(['Label', 'Metric'])
    for label in ['Macro', 'Pneumonia', 'Pneumothorax']:
        for metric in ['AUROC', 'AUPRC']:
            row = table.loc[label]
            ci = intervals.loc[(label, metric)]
            low, high = ci['CI95_low'], ci['CI95_high']
            if ci['N_bootstrap'] < 0.9 * BOOTSTRAP_REPEATS or not np.isfinite([low, high]).all():
                verdict = 'Không đủ bootstrap hợp lệ'
            elif low > 0:
                verdict = 'ASL cao hơn; CI không chứa 0'
            elif high < 0:
                verdict = 'ASL thấp hơn; CI không chứa 0'
            else:
                verdict = 'Chưa rõ khác biệt; CI chứa 0'
            report.append(
                f'| {label} | {metric} | {row[f"{metric}_BCE"]:.5f} | {row[f"{metric}_ASL"]:.5f} | '
                f'{row[f"Delta_{metric}"]:+.5f} | [{low:+.5f}, {high:+.5f}] | {verdict} |'
            )
    report += ['', 'Precision/Recall/Specificity/F1 và TP/FP/TN/FN từng nhãn nằm trong `comparison_asl_vs_bce.csv`.', '']
report += [
    '## Diễn giải', '',
    'Ưu tiên Macro AUROC và AUROC/AUPRC của Pneumonia/Pneumothorax. '
    'Xem cả precision và recall tại threshold development, vì ASL có thể đổi calibration.',
    'CI bootstrap theo bệnh nhân là CI mô tả cho hai checkpoint cố định, chưa điều chỉnh nhiều phép so sánh. '
    'Một training seed không chứng minh ASL ổn định hơn BCE; cần train nhiều seed cho kết luận đó.',
    'Không so sánh độ lớn training loss ASL với BCE và không tune trên tập đánh giá.', '',
]
report_text = '\n'.join(report)
(OUT_DIR / 'comparison_report.md').write_text(report_text, encoding='utf-8')
print(report_text)
print('\nArtifacts:', OUT_DIR.resolve())
